# 05 - Vector Similarity Retrieval, Chunk-Level (ContraDoc)

Under the GraphRAG schema from step 03, candidate contradiction pairs are **chunk pairs**. Each `:Chunk` carries its sentence text and a pre-computed SBERT embedding. This notebook retrieves per-document top-K chunk pairs by cosine similarity and reports gold-pair recall alongside the structural baselines from step 04.

## Retrieval

For each document, every intra-doc chunk pair is scored by cosine similarity on the chunk embeddings, and the K highest-scoring pairs are kept. Pair count scales as `K × #docs` exactly (e.g., K=10 → 1,000 pairs across 100 docs).

Structural retrieval (S-SR / S-SO / S-Union) is unchanged — the `:RELATION` subgraph from step 03 still powers both patterns, aggregated to unique chunk pairs via their `sentence_id`s.

**Output:** `data/processed/ContraDoc/chunk_candidates.jsonl` — one JSON object per candidate chunk pair at the chosen K (unioned with S-Union), consumed by notebook 06 (NLI).

In [1]:
import json
from collections import defaultdict
from pathlib import Path

import numpy as np
from neo4j import GraphDatabase

from config import settings

OUTPUT_PATH = Path("data/processed/ContraDoc/chunk_candidates.jsonl")
K_VALUES = [1, 3, 5, 10, 20]
PRIMARY_K = 20

## Connect and pull chunks from Neo4j

Each `:Chunk` node carries the sentence text and its pre-computed embedding from step 03.

In [2]:
driver = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value()),
)
driver.verify_connectivity()


def run(cypher, **params):
    with driver.session() as s:
        return list(s.run(cypher, **params))


chunk_rows = run(
    "MATCH (c:Chunk) "
    "RETURN c.doc_id AS doc_id, c.sentence_id AS sid, c.source_text AS text, c.embedding AS embedding, "
    "       c.is_gold_evidence AS is_gold_evidence, c.is_gold_ref AS is_gold_ref, c.ref_index AS ref_index"
)
chunks = [dict(r) for r in chunk_rows]
print(f"Loaded {len(chunks)} chunks from Neo4j.")

doc_label = {r["d"]: r["l"] for r in run("MATCH (d:Document) RETURN d.doc_id AS d, d.contradiction AS l")}
print(f"YES docs: {sum(1 for v in doc_label.values() if v == 'YES')}, NO docs: {sum(1 for v in doc_label.values() if v == 'NO')}")

Loaded 3664 chunks from Neo4j.
YES docs: 50, NO docs: 50


## Chunk keys, gold pairs, and the pkey helper

A chunk is identified by `(doc_id, sentence_id)`. A chunk pair is an unordered tuple of two such keys.

In [3]:
def chunk_key(c):
    return (c["doc_id"], c["sid"])


def pkey(a, b):
    return (a, b) if a < b else (b, a)


# Build per-doc chunk lists + index arrays for the embedding matrix
chunks_by_doc = defaultdict(list)
for i, c in enumerate(chunks):
    chunks_by_doc[c["doc_id"]].append(i)

# Stack embeddings into a matrix
embeddings = np.asarray([c["embedding"] for c in chunks], dtype=np.float32)
print(f"Embedding matrix shape: {embeddings.shape}")

# Gold chunk pairs: for each YES doc, (evidence chunk, each ref chunk) within the same doc
gold_pairs = set()
for ev in chunks:
    if not ev["is_gold_evidence"]:
        continue
    for ref in chunks:
        if ref["is_gold_ref"] and ref["doc_id"] == ev["doc_id"]:
            gold_pairs.add(pkey(chunk_key(ev), chunk_key(ref)))

n_gold_usable_docs = len({k[0][0] for k in gold_pairs})
print(f"Gold chunk pairs: {len(gold_pairs)} across {n_gold_usable_docs} docs")

Embedding matrix shape: (3664, 384)
Gold chunk pairs: 37 across 36 docs


## Structural baselines (S-SR / S-SO / S-Union), aggregated to chunk pairs

The Cypher patterns are unchanged from step 04 — they find **triple** pairs. To align with the chunk-level output format of steps 05/06, we aggregate each matched `(triple_a, triple_b)` to a chunk pair `(sentence_id_a, sentence_id_b)` and DISTINCT to remove duplicates.

In [4]:
S_SR_CHUNK = """
MATCH (s:Entity)-[r1:RELATION]->(o1:Entity), (s)-[r2:RELATION]->(o2:Entity)
WHERE r1.doc_id = r2.doc_id AND r1.predicate = r2.predicate
  AND elementId(r1) < elementId(r2)
  AND o1.name <> o2.name
  AND r1.sentence_id <> r2.sentence_id
RETURN DISTINCT r1.doc_id AS doc_id, r1.sentence_id AS sid_a, r2.sentence_id AS sid_b
"""

S_SO_CHUNK = """
MATCH (s:Entity)-[r1:RELATION]->(o:Entity), (s)-[r2:RELATION]->(o)
WHERE r1.doc_id = r2.doc_id AND r1.predicate <> r2.predicate
  AND elementId(r1) < elementId(r2)
  AND r1.sentence_id <> r2.sentence_id
RETURN DISTINCT r1.doc_id AS doc_id, r1.sentence_id AS sid_a, r2.sentence_id AS sid_b
"""


def rows_to_chunk_pairs(rows):
    return {pkey((r["doc_id"], r["sid_a"]), (r["doc_id"], r["sid_b"])) for r in rows}


s_sr = rows_to_chunk_pairs(run(S_SR_CHUNK))
s_so = rows_to_chunk_pairs(run(S_SO_CHUNK))
s_union = s_sr | s_so

print(f"S-SR  chunk pairs: {len(s_sr)}")
print(f"S-SO  chunk pairs: {len(s_so)}")
print(f"S-Union chunk pairs: {len(s_union)}")

S-SR  chunk pairs: 459
S-SO  chunk pairs: 109
S-Union chunk pairs: 565


## Vector retrieval — top-K pairs per document

For each document, score every intra-doc unordered chunk pair by cosine similarity and keep the K highest-scoring pairs. Total pair count = `K × #docs`.

In [5]:
def vector_pairs(k: int) -> set[tuple]:
    """Top-K pairs per doc: rank every intra-doc chunk pair by cosine, keep the K highest per doc."""
    pairs: set[tuple] = set()
    for doc_id, idxs in chunks_by_doc.items():
        if len(idxs) < 2:
            continue
        emb = embeddings[idxs]
        sim = emb @ emb.T
        n = len(idxs)
        scored = []
        for li in range(n):
            for lj in range(li + 1, n):
                scored.append((float(sim[li, lj]), idxs[li], idxs[lj]))
        scored.sort(key=lambda x: -x[0])
        for _, gi, gj in scored[:k]:
            pairs.add(pkey(chunk_key(chunks[gi]), chunk_key(chunks[gj])))
    return pairs


vec_by_k = {k: vector_pairs(k) for k in K_VALUES}
print("Vector pair counts per K:")
for k in K_VALUES:
    print(f"  K={k:>3}: {len(vec_by_k[k])} pairs")

Vector pair counts per K:
  K=  1: 100 pairs
  K=  3: 300 pairs
  K=  5: 500 pairs
  K= 10: 1000 pairs
  K= 20: 2000 pairs


## Ablation: gold-pair recall + per-doc volume

Pair-recall = caught gold / total gold chunk pairs. Doc-recall = docs with ≥ 1 caught gold / `n_gold_usable_docs`. Precision = caught gold / candidates.

In [6]:
def metrics(pairs: set[tuple], name: str) -> dict:
    caught = pairs & gold_pairs
    docs_caught = {p[0][0] for p in caught}
    per_doc = defaultdict(int)
    for a, _ in pairs:
        per_doc[a[0]] += 1
    yes_vol = [per_doc.get(d, 0) for d, l in doc_label.items() if l == "YES"]
    no_vol = [per_doc.get(d, 0) for d, l in doc_label.items() if l == "NO"]
    return {
        "name": name,
        "n": len(pairs),
        "caught": len(caught),
        "pair_r": len(caught) / max(len(gold_pairs), 1),
        "doc_r": len(docs_caught) / max(n_gold_usable_docs, 1),
        "docs_caught": len(docs_caught),
        "prec": len(caught) / max(len(pairs), 1),
        "yes_mean": sum(yes_vol) / max(len(yes_vol), 1),
        "no_mean": sum(no_vol) / max(len(no_vol), 1),
    }


rows = [
    metrics(s_sr, "S-SR"),
    metrics(s_so, "S-SO"),
    metrics(s_union, "S-Union"),
]
for k in K_VALUES:
    rows.append(metrics(vec_by_k[k], f"Vector@{k}"))
    rows.append(metrics(s_union | vec_by_k[k], f"Vector+S@{k}"))

print(f"Gold chunk pairs in graph: {len(gold_pairs)} across {n_gold_usable_docs} docs")
print()
header = f"{'Method':14} {'#cand':>7}  {'caught':>6}  {'Pair-R':>7}  {'Doc-R':>11}  {'Prec':>6}  {'YES mean':>9}  {'NO mean':>9}"
print(header)
print("-" * len(header))
for r in rows:
    print(
        f"{r['name']:14} {r['n']:>7}  {r['caught']:>6}  "
        f"{r['pair_r']:>6.1%}  "
        f"{r['docs_caught']:>2}/{n_gold_usable_docs:<2} {r['doc_r']:>5.1%}  "
        f"{r['prec']:>5.1%}  "
        f"{r['yes_mean']:>8.1f}  {r['no_mean']:>8.1f}"
    )

Gold chunk pairs in graph: 37 across 36 docs

Method           #cand  caught   Pair-R        Doc-R    Prec   YES mean    NO mean
----------------------------------------------------------------------------------
S-SR               459      12   32.4%  12/36 33.3%   2.6%       3.8       5.4
S-SO               109       4   10.8%   4/36 11.1%   3.7%       1.1       1.1
S-Union            565      16   43.2%  16/36 44.4%   2.8%       4.8       6.5
Vector@1           100      13   35.1%  13/36 36.1%  13.0%       1.0       1.0
Vector+S@1         650      20   54.1%  20/36 55.6%   3.1%       5.5       7.5
Vector@3           300      15   40.5%  15/36 41.7%   5.0%       3.0       3.0
Vector+S@3         832      20   54.1%  20/36 55.6%   2.4%       7.4       9.3
Vector@5           500      17   45.9%  17/36 47.2%   3.4%       5.0       5.0
Vector+S@5        1018      21   56.8%  21/36 58.3%   2.1%       9.3      11.1
Vector@10         1000      20   54.1%  20/36 55.6%   2.0%      10.0      10.

## Save the primary candidate set for step 06

Default: Vector@K ∪ S-Union at `PRIMARY_K` (set at the top of this notebook).

In [7]:
primary_vec = vec_by_k[PRIMARY_K]
primary = s_union | primary_vec
print(f"Primary: Vector@{PRIMARY_K} ∪ S-Union = {len(primary)} candidate chunk pairs")

chunk_by_key = {chunk_key(c): c for c in chunks}

# Compute cosine similarity for each pair (for step 06 reference)
sim_lookup: dict[tuple, float] = {}
for doc_id, idxs in chunks_by_doc.items():
    if len(idxs) < 2:
        continue
    emb = embeddings[idxs]
    sim = emb @ emb.T
    for li in range(len(idxs)):
        for lj in range(li + 1, len(idxs)):
            key = pkey(chunk_key(chunks[idxs[li]]), chunk_key(chunks[idxs[lj]]))
            sim_lookup[key] = float(sim[li, lj])

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    for key_a, key_b in primary:
        ca, cb = chunk_by_key[key_a], chunk_by_key[key_b]
        in_struct = (key_a, key_b) in s_union
        in_vec = (key_a, key_b) in primary_vec
        source = "struct+vector" if (in_struct and in_vec) else ("struct" if in_struct else "vector")
        rec = {
            "doc_id": ca["doc_id"],
            "source": source,
            "similarity": sim_lookup.get((key_a, key_b)),
            "is_gold_pair": (key_a, key_b) in gold_pairs,
            "chunk_a": {
                "sentence_id": ca["sid"],
                "source_text": ca["text"],
                "is_gold_evidence": ca["is_gold_evidence"],
                "is_gold_ref": ca["is_gold_ref"],
            },
            "chunk_b": {
                "sentence_id": cb["sid"],
                "source_text": cb["text"],
                "is_gold_evidence": cb["is_gold_evidence"],
                "is_gold_ref": cb["is_gold_ref"],
            },
        }
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Saved {len(primary)} chunk-pair candidates -> {OUTPUT_PATH.resolve()}")

# Sample gold pair caught only by vector (not structural)
vec_only_gold = (primary_vec & gold_pairs) - s_union
if vec_only_gold:
    key_a, key_b = next(iter(vec_only_gold))
    ca, cb = chunk_by_key[key_a], chunk_by_key[key_b]
    print()
    print("Sample gold chunk pair caught by Vector but NOT by S-Union:")
    print(f"  doc={ca['doc_id']}  similarity={sim_lookup.get((key_a, key_b)):.3f}")
    print(f"  A [sid={ca['sid']}] (ev={ca['is_gold_evidence']} ref={ca['is_gold_ref']}): {ca['text'][:140]}")
    print(f"  B [sid={cb['sid']}] (ev={cb['is_gold_evidence']} ref={cb['is_gold_ref']}): {cb['text'][:140]}")

driver.close()

Primary: Vector@20 ∪ S-Union = 2451 candidate chunk pairs
Saved 2451 chunk-pair candidates -> D:\AT82.05-Claim-Contradiction-Over-Knowledge-Graphs\experiments\data\processed\ContraDoc\chunk_candidates.jsonl

Sample gold chunk pair caught by Vector but NOT by S-Union:
  doc=3489738277_3  similarity=0.479
  A [sid=4] (ev=False ref=True): In Louisville, Kentucky, Sen. Rand Paul made the not-so-surprising announcement that he will run for president, while in Chicago, voters wil
  B [sid=16] (ev=True ref=False): OK, sure, everyone was shocked when the Kentucky senator announced his bid for the Oval Office, but of course it was news when he made it of


## Per-type recall breakdown

Per ContraDoc contradiction type — which retrieval strategies recover which types? Multi-label docs (e.g., `Content|Numeric`) contribute to every listed type.

In [8]:
driver_t = GraphDatabase.driver(
    settings.neo4j_uri,
    auth=(settings.neo4j_user, settings.neo4j_password.get_secret_value()),
)
with driver_t.session() as _s:
    doc_types = {
        r["doc_id"]: [t for t in (r["contra_type"] or "none").split("|") if t]
        for r in _s.run("MATCH (d:Document {contradiction: 'YES'}) RETURN d.doc_id AS doc_id, d.contra_type AS contra_type")
    }
driver_t.close()


def per_type_recall(pairs, name):
    type_totals = defaultdict(int)
    type_caught = defaultdict(int)
    for p in gold_pairs:
        doc_id = p[0][0]
        for t in doc_types.get(doc_id, ["unknown"]):
            type_totals[t] += 1
            if p in pairs:
                type_caught[t] += 1
    all_types = sorted(type_totals.keys(), key=lambda x: -type_totals[x])
    print(f"\n{name}:")
    print(f"  {'type':30s}  caught  total  recall")
    print("  " + "-" * 52)
    for t in all_types:
        caught, total = type_caught[t], type_totals[t]
        print(f"  {t:30s}  {caught:>6}  {total:>5}  {caught / max(total, 1):>6.1%}")


per_type_recall(s_sr, "S-SR")
per_type_recall(s_so, "S-SO")
per_type_recall(s_union, "S-Union")
for k in K_VALUES:
    per_type_recall(vec_by_k[k], f"Vector@{k}")
    per_type_recall(s_union | vec_by_k[k], f"Vector+S@{k}")


S-SR:
  type                            caught  total  recall
  ----------------------------------------------------
  Content                             11     23   47.8%
  Numeric                              3     10   30.0%
  Negation                             0      6    0.0%
  Perspective/View/Opinion             2      6   33.3%
  Factual                              1      5   20.0%
  Emotion/Mood/Feeling                 0      4    0.0%
  Causal                               0      2    0.0%
  Relation                             0      1    0.0%

S-SO:
  type                            caught  total  recall
  ----------------------------------------------------
  Content                              3     23   13.0%
  Numeric                              1     10   10.0%
  Negation                             2      6   33.3%
  Perspective/View/Opinion             2      6   33.3%
  Factual                              0      5    0.0%
  Emotion/Mood/Feeling              